# Incremental Learning — Phase 2: Adaptive KS Reference (v2, WINDOW=60)

**Goal**: stop the model from firing fine-tunes at every check (was 15/15) by
updating the KS reference after each successful fine-tune, and require
**two consecutive** significant KS hits before fine-tuning.

**Also**: use Phase-1 FPR-targeted threshold during streaming (falls back to
mean+k·std if Phase-1 artifact missing `threshold_mode`).

Control = Phase-1 threshold, incremental learning OFF.
Treatment = Phase-1 threshold + reference-updating incremental learning.


In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import pickle, joblib, os, copy, time
import matplotlib.pyplot as plt
from pathlib import Path
from collections import deque
from scipy import stats
from sklearn.metrics import (
    precision_score, recall_score, f1_score, confusion_matrix,
    roc_auc_score, average_precision_score, precision_recall_curve,
)

def resolve_base():
    here = Path.cwd().resolve()
    candidates = [
        here if here.name == 'module3' else None,
        here.parent if here.name == 'module3_pipeline_v2' else None,
        Path(r'c:\Users\DELL\Documents\Claude\Projects\FYP\module3'),
        Path(r'c:\Users\jthar\Documents\Claude\Projects\module3\module3'),
    ]
    for c in candidates:
        if c is not None and (c / 'models_v2').exists():
            return str(c)
    raise FileNotFoundError('Could not locate module3 BASE (models_v2 missing).')

BASE = resolve_base()
DATA_DIR  = os.path.join(BASE, 'data', 'processed')
WIN_DIR   = os.path.join(BASE, 'data', 'processed', 'windows_cc1_v2')
WIN_DIR_V1 = os.path.join(BASE, 'data', 'processed', 'windows_cc1')
MODEL_DIR = os.path.join(BASE, 'models_v2')
MODEL_DIR_V1 = os.path.join(BASE, 'models')
print('BASE =', BASE)

WINDOW_SIZE = 60
BUFFER_SIZE = 500
REFIT_INTERVAL = 5000
FT_BUFFER_SIZE = 2000
FT_LR = 1e-4
FT_EPOCHS = 5
KS_ALPHA = 0.001
KS_STREAK_REQUIRED = 2          # Phase-2: require consecutive drift signals
REF_UPDATE_MODE = 'replace'    # 'replace' | 'ema'
REF_EMA_ALPHA = 0.5             # used only if REF_UPDATE_MODE == 'ema'
TARGET_FPR = 0.01
MIN_LOCAL_FOR_PERCENTILE = 30
print('Phase 2 constants ready.')


## Step 1 — Load model + Phase-1 threshold config


In [ ]:
class VAE(nn.Module):
    def __init__(self, input_dim, hidden1, hidden2, latent_dim):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden1), nn.ReLU(),
            nn.Linear(hidden1, hidden2), nn.ReLU(),
        )
        self.fc_mu = nn.Linear(hidden2, latent_dim)
        self.fc_lv = nn.Linear(hidden2, latent_dim)
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden2), nn.ReLU(),
            nn.Linear(hidden2, hidden1), nn.ReLU(),
            nn.Linear(hidden1, input_dim),
        )

    def encode(self, x):
        h = self.encoder(x)
        return self.fc_mu(h), torch.clamp(self.fc_lv(h), -10, 10)

    def decode(self, z):
        return self.decoder(z)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        return mu + torch.randn_like(std) * std

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        return self.decode(z), mu, logvar

    @torch.no_grad()
    def anomaly_score(self, x):
        self.eval()
        mu, _ = self.encode(x)
        return ((self.decode(mu) - x) ** 2).mean(dim=1)

meta = pickle.load(open(os.path.join(MODEL_DIR, 'vae_cc1_meta.pkl'), 'rb'))
static_eval = pickle.load(open(os.path.join(MODEL_DIR, 'vae_cc1_eval.pkl'), 'rb'))
adaptive_cfg = pickle.load(open(os.path.join(MODEL_DIR, 'vae_cc1_adaptive_blended_eval.pkl'), 'rb'))

CLIP = meta['clip']
GLOBAL_MEAN = float(meta['mu_train'])
GLOBAL_STD = float(meta['sigma_train'])
STATIC_VAL_P99 = float(static_eval['thresholds']['val_p99'])
K_ADAPTIVE = float(adaptive_cfg.get('k_adaptive', (STATIC_VAL_P99 - GLOBAL_MEAN) / GLOBAL_STD))
PRIOR_STRENGTH = int(adaptive_cfg['prior_strength'])
THRESHOLD_MODE = adaptive_cfg.get('threshold_mode', 'mean_k')  # 'fpr_targeted' after Phase 1

base_model = VAE(meta['input_dim'], meta['hidden1'], meta['hidden2'], meta['latent_dim'])
base_model.load_state_dict(torch.load(os.path.join(MODEL_DIR, 'vae_cc1.pt'), map_location='cpu'))
base_model.eval()

n_params = sum(p.numel() for p in base_model.parameters())
file_size_kb = os.path.getsize(os.path.join(MODEL_DIR, 'vae_cc1.pt')) / 1024
print(f'THRESHOLD_MODE={THRESHOLD_MODE}  PRIOR_STRENGTH={PRIOR_STRENGTH}  K_ADAPTIVE={K_ADAPTIVE:.4f}')
print(f'Params={n_params:,}  size={file_size_kb:.1f} KB')

X_cc1_train = np.clip(np.load(os.path.join(WIN_DIR, 'X_cc1_train.npy')), -CLIP, CLIP).astype(np.float32)
rng = np.random.default_rng(42)
REF_IDX = rng.choice(len(X_cc1_train), size=min(5000, len(X_cc1_train)), replace=False)
REF_SAMPLE = X_cc1_train[REF_IDX]
with torch.no_grad():
    REFERENCE_MSE_FROZEN = base_model.anomaly_score(torch.from_numpy(REF_SAMPLE)).numpy().copy()
print(f'Frozen KS reference: {len(REFERENCE_MSE_FROZEN):,} windows')


## Step 2 — Rebuild drift_cc2 stream in chronological order


In [ ]:
FEATURE_COLS = [
    'container_cpu_usage_seconds_rate', 'container_cpu_system_seconds_rate', 'container_cpu_user_seconds_rate',
    'container_memory_usage_bytes', 'container_memory_working_set_bytes',
    'container_memory_rss', 'container_memory_cache',
]

def window_meta_with_features(df, feature_cols, window_size=WINDOW_SIZE, stride=1):
    X, cmdb_ids, end_ts, ys, fts = [], [], [], [], []
    for cmdb_id, g in df.sort_values('timestamp').groupby('cmdb_id'):
        data = g[feature_cols].values.astype(np.float32)
        is_gap = g['is_gap'].values
        labels = g['label'].values
        ftypes = g['failure_type'].values.astype(object)
        ts = g['timestamp'].values
        n = len(g)
        for i in range(0, n - window_size + 1, stride):
            if is_gap[i:i + window_size].any():
                continue
            X.append(data[i:i + window_size])
            cmdb_ids.append(cmdb_id)
            end_ts.append(ts[i + window_size - 1])
            ys.append(int(labels[i:i + window_size].any()))
            w_types = sorted({t for t in ftypes[i:i + window_size] if isinstance(t, str)})
            fts.append(','.join(w_types) if w_types else None)
    return (np.stack(X), np.array(cmdb_ids), np.array(end_ts),
            np.array(ys, dtype=np.int64), np.array(fts, dtype=object))

df_cc2 = pd.read_csv(os.path.join(DATA_DIR, 'drift_complex_case2.csv'), low_memory=False)
df_cc2['is_gap'] = df_cc2['is_gap'].astype(bool)
X_raw, cmdb_ids, end_ts, y_true, ft_true = window_meta_with_features(df_cc2, FEATURE_COLS)
y_saved = np.load(os.path.join(WIN_DIR, 'y_drift_cc2.npy'))
assert np.array_equal(y_true, y_saved), 'window rebuild mismatch'

pca = joblib.load(os.path.join(MODEL_DIR, 'cc1_pca.pkl'))['pca']
X_pca = np.clip(pca.transform(X_raw.reshape(len(X_raw), -1)), -CLIP, CLIP).astype(np.float32)

order = np.argsort(end_ts, kind='stable')
X_stream = X_pca[order]
y_stream = y_true[order]
ft_stream = ft_true[order]
cmdb_stream = cmdb_ids[order]
print(f'Stream ready: {len(X_stream):,} windows, {int(y_stream.sum())} anomalies')


## Step 3 — Streaming simulator with Phase-1 threshold + Phase-2 reference update


In [ ]:
def decide_threshold(mse, buf, mode=THRESHOLD_MODE):
    n_local = len(buf)
    w = n_local / (n_local + PRIOR_STRENGTH)
    if mode == 'fpr_targeted':
        if n_local < MIN_LOCAL_FOR_PERCENTILE:
            local_t = STATIC_VAL_P99
        else:
            local_t = float(np.percentile(np.fromiter(buf, dtype=np.float64), (1 - TARGET_FPR) * 100))
        return w * local_t + (1 - w) * STATIC_VAL_P99
    # mean+k fallback
    if n_local == 0:
        lm, ls = GLOBAL_MEAN, GLOBAL_STD
    else:
        arr = np.fromiter(buf, dtype=np.float64)
        lm, ls = arr.mean(), (arr.std() if n_local > 1 else GLOBAL_STD)
    return (w * lm + (1 - w) * GLOBAL_MEAN) + K_ADAPTIVE * (w * ls + (1 - w) * GLOBAL_STD)


def run_stream(model, X_stream, cmdb_stream, incremental_learning,
               reference_mse_init, update_reference=True, streak_required=KS_STREAK_REQUIRED,
               verbose=True):
    model = copy.deepcopy(model)
    opt = torch.optim.Adam(model.parameters(), lr=FT_LR)
    reference_mse = reference_mse_init.copy()

    preds = np.zeros(len(X_stream), dtype=np.int64)
    mse_trace = np.zeros(len(X_stream), dtype=np.float64)
    threshold_buffers = {}
    ft_pool = deque(maxlen=FT_BUFFER_SIZE)
    n_finetunes = 0
    finetune_events = []
    ks_events = []
    drift_streak = 0

    for i in range(len(X_stream)):
        x_i = X_stream[i]
        cid = cmdb_stream[i]
        with torch.no_grad():
            mse = model.anomaly_score(torch.from_numpy(x_i).unsqueeze(0)).item()
        mse_trace[i] = mse

        buf = threshold_buffers.setdefault(cid, deque(maxlen=BUFFER_SIZE))
        t = decide_threshold(mse, buf)
        is_anom = mse > t
        preds[i] = int(is_anom)
        if not is_anom:
            buf.append(mse)
            ft_pool.append(x_i)

        if incremental_learning and (i + 1) % REFIT_INTERVAL == 0 and len(ft_pool) >= FT_BUFFER_SIZE // 2:
            with torch.no_grad():
                recent_mse = model.anomaly_score(torch.from_numpy(np.stack(list(ft_pool)))).numpy()
            ks_stat, p_value = stats.ks_2samp(reference_mse, recent_mse)
            drifted = p_value < KS_ALPHA
            ks_events.append({'i': i, 'ks': float(ks_stat), 'p': float(p_value), 'drifted': bool(drifted)})
            if drifted:
                drift_streak += 1
            else:
                drift_streak = 0

            if drifted and drift_streak >= streak_required:
                Xb = torch.from_numpy(np.stack(list(ft_pool)))
                model.train()
                for _ in range(FT_EPOCHS):
                    opt.zero_grad()
                    recon, mu, logvar = model(Xb)
                    recon_loss = nn.functional.mse_loss(recon, Xb, reduction='mean')
                    kl = -0.5 * torch.mean(torch.sum(1 + logvar - mu.pow(2) - logvar.exp(), dim=1))
                    (recon_loss + meta['beta_max'] * kl).backward()
                    opt.step()
                model.eval()
                n_finetunes += 1
                finetune_events.append({'i': i, 'p': float(p_value), 'streak': drift_streak})
                drift_streak = 0

                if update_reference:
                    with torch.no_grad():
                        adapted = model.anomaly_score(torch.from_numpy(np.stack(list(ft_pool)))).numpy()
                    # Resize adapted sample to reference length via bootstrap
                    rng_i = np.random.default_rng(i)
                    adapted_ref = adapted[rng_i.choice(len(adapted), size=len(reference_mse), replace=True)]
                    if REF_UPDATE_MODE == 'ema':
                        reference_mse = (REF_EMA_ALPHA * adapted_ref +
                                         (1 - REF_EMA_ALPHA) * reference_mse)
                    else:
                        reference_mse = adapted_ref
                    if verbose:
                        print(f'  [window {i+1:>6}] fine-tune #{n_finetunes} (p={p_value:.2e}, streak ok) '
                              f'+ reference {REF_UPDATE_MODE}')
                elif verbose:
                    print(f'  [window {i+1:>6}] fine-tune #{n_finetunes} (p={p_value:.2e}) [no ref update]')
            elif verbose and drifted:
                print(f'  [window {i+1:>6}] drift signal (p={p_value:.2e}) streak={drift_streak}/{streak_required} — wait')

    return preds, mse_trace, n_finetunes, finetune_events, ks_events

print('Simulator ready.')


## Step 4 — Control vs treatment (+ legacy frozen-reference ablation)


In [ ]:
def report(name, preds):
    p = precision_score(y_stream, preds, zero_division=0)
    r = recall_score(y_stream, preds, zero_division=0)
    f1 = f1_score(y_stream, preds, zero_division=0)
    tn, fp, fn, tp = confusion_matrix(y_stream, preds).ravel()
    fpr = fp / (fp + tn) if (fp + tn) else float('nan')
    print(f'{name:28s} P={p:.3f}  R={r:.3f}  F1={f1:.3f}  FPR={fpr:.4f}  FP={fp}')
    return {'precision': float(p), 'recall': float(r), 'f1': float(f1), 'fpr': float(fpr),
            'tp': int(tp), 'fp': int(fp), 'fn': int(fn), 'tn': int(tn)}

print('CONTROL (IL OFF)...')
preds_c, _, nft_c, _, _ = run_stream(
    base_model, X_stream, cmdb_stream, incremental_learning=False,
    reference_mse_init=REFERENCE_MSE_FROZEN, update_reference=False, verbose=False)
assert nft_c == 0
result_control = report('control', preds_c)

print('\nTREATMENT Phase-2 (IL ON, ref update, streak=2)...')
preds_t, _, nft_t, events_t, ks_t = run_stream(
    base_model, X_stream, cmdb_stream, incremental_learning=True,
    reference_mse_init=REFERENCE_MSE_FROZEN, update_reference=True,
    streak_required=KS_STREAK_REQUIRED, verbose=True)
result_treat = report('treatment_phase2', preds_t)
print(f'n_finetunes={nft_t}  (legacy frozen-ref was 15)')

print('\nABLATION: IL ON but frozen reference + streak=1 (legacy behaviour)...')
preds_legacy, _, nft_legacy, _, _ = run_stream(
    base_model, X_stream, cmdb_stream, incremental_learning=True,
    reference_mse_init=REFERENCE_MSE_FROZEN, update_reference=False,
    streak_required=1, verbose=False)
result_legacy = report('legacy_frozen_ref', preds_legacy)
print(f'n_finetunes_legacy={nft_legacy}')

print(f'\nF1 change Phase2 vs control: {result_treat["f1"] - result_control["f1"]:+.3f}')
print(f'FPR change Phase2 vs control: {result_treat["fpr"] - result_control["fpr"]:+.4f}')


## Step 5 — Save


In [ ]:
save_results = {
    'phase': 2,
    'lightweight_validation': {
        'n_params': n_params, 'file_size_kb': file_size_kb,
    },
    'config': {
        'refit_interval': REFIT_INTERVAL, 'ft_buffer_size': FT_BUFFER_SIZE,
        'ft_lr': FT_LR, 'ft_epochs': FT_EPOCHS, 'ks_alpha': KS_ALPHA,
        'ks_streak_required': KS_STREAK_REQUIRED,
        'ref_update_mode': REF_UPDATE_MODE, 'ref_ema_alpha': REF_EMA_ALPHA,
        'threshold_mode': THRESHOLD_MODE, 'prior_strength': PRIOR_STRENGTH,
    },
    'incremental_learning': {
        'n_finetunes': nft_t, 'finetune_events': events_t, 'ks_events': ks_t,
        'control': result_control,
        'treatment': result_treat,
        'legacy_frozen_ref': {**result_legacy, 'n_finetunes': nft_legacy},
    },
}
out_path = os.path.join(MODEL_DIR, 'incremental_learning_eval.pkl')
with open(out_path, 'wb') as f:
    pickle.dump(save_results, f)
print(f'Saved -> {out_path}')

phase2_path = os.path.join(MODEL_DIR, 'phase2_incremental_ref_update_eval.pkl')
with open(phase2_path, 'wb') as f:
    pickle.dump(save_results, f)
print(f'Saved -> {phase2_path}')


## Phase 2 summary

- Control uses Phase-1 FPR-targeted threshold only.
- Treatment adds KS-triggered fine-tuning with **reference refresh** and **streak=2**.
- Legacy ablation keeps frozen CC1 reference for attribution.
